Dependencies

In [121]:
import pandas as pd
import numpy as np
import statistics as sts
import matplotlib as plt
import seaborn as sns
rng = np.random.default_rng(12345)

I have also made use of the datawrangler extension to visualize the dataframes and get an idea of general missingness/unique id and whether ranges of numeric values are plausible. I have tried to also incorporate it into the code here where possible but I wanted to inform you of this in the Dependencies.

Loading the data

In [122]:
dfGenes = pd.read_csv("../data/genes.csv")
dfNL = pd.read_csv("../data/NL.csv")
dfPL = pd.read_csv("../data/PL.csv")
dfUK = pd.read_csv("../data/UK.csv")
dfUS = pd.read_csv("../data/US.csv")
dfgenes = pd.read_csv("../data/genes.csv")

## Initial preprocessing of NL data

In [123]:
dfNL
dfNL["height"] = dfNL["height"].astype("Int64")
dfNL["weight"] = dfNL["weight"].astype("Int64")
dfNL["BMI"] = (dfNL.weight)/(dfNL.height/100)**2

 This is the standard to which the others must match, however since the other dataframes have some missing values in some columns, the dtype of the NL dataframe for height and weight of int64 will not work since I cannot encode missing values in those. As such I updated it to Int64 for merging later. Also added BMI column


## Preprocessing of PL data


In [124]:
dfPL # So missing values, misnamed height column, smoke status values differ and decimals in numeric variables, order is also wrong of columns. Lets fix it!

,id,heigth,weight,sex,smokes
0,S0001,178.0,77.0,female,False
1,S0002,155.0,74.0,male,False
2,S0003,173.0,70.0,male,True
3,S0004,167.0,95.0,male,False
4,S0005,181.0,78.0,female,False
...,...,...,...,...,...
1533,S0537,158.0,56.0,male,False
1534,S0538,159.0,43.0,male,False
1535,S0539,162.0,80.0,male,False
1536,S053a,194.0,110.0,NaN,True


In [125]:
dfPL["id"].is_unique # There are some ids that are not unique
dupes = dfPL[dfPL["id"].duplicated()] # These are the duplicated ids, its id "S030f" duplicated 199 times from row 783 to 981
dfPL = dfPL.drop(dupes.index).reset_index(drop=True) # Run once to drop these duplicated rows whilst reseting index




In [126]:
dfPL.loc[(dfPL.smokes == False), "smokes"] = "no" # Convert the smoking variable to the dutch standard
dfPL.loc[(dfPL.smokes == True), "smokes"] = "yes"

dfPL["smokes"] = dfPL["smokes"].astype(str) # Just so all dataframes have the same object type for each column

In [127]:
dfPL = dfPL.rename(columns={"heigth": "height"}) # Rename the height column to match dutch standard

In [128]:
dfPL["height"] = dfPL["height"].round(0).astype("Int64") # Getting rid of decimals in the numeric variables to match dutch standard, had to use round(0) and astype("Int64") instead of just int because of missing values
dfPL["weight"] = dfPL["weight"].round(0).astype("Int64")

In [129]:
dfPL = dfPL.loc[:, ["id", "weight", "height", "sex", "smokes"]] # Reordering the columns to match dutch standard

In [130]:
dfPL["BMI"] = (dfPL.weight)/(dfPL.height/100)**2 # Calculating BMI for polish data and adding to DF
dfPL # Looks good!

,id,weight,height,sex,smokes,BMI
0,S0001,77,178,female,no,24.302487
1,S0002,74,155,male,no,30.801249
2,S0003,70,173,male,yes,23.388687
3,S0004,95,167,male,no,34.063609
4,S0005,78,181,female,no,23.808797
...,...,...,...,...,...,...
1334,S0537,56,158,male,no,22.432303
1335,S0538,43,159,male,no,17.008821
1336,S0539,80,162,male,no,30.483158
1337,S053a,110,194,NaN,yes,29.227336


## Preprocessing of US data

In [131]:
dfUS["id"].is_unique # US data ids are unique
dfUS # Only issue seems to be missing values which I elected to keep in the data for now instead of deleting rows with missing value. Ill turn height into whole number again:


,id,weight,height,sex,smokes
0,S1272,61,NaN,female,no
1,S1288,105,NaN,female,no
2,S1361,92,NaN,female,no
3,S1091,78,182.0,male,no
4,S1351,83,NaN,female,no
...,...,...,...,...,...
456,S1300,61,NaN,female,no
457,S1169,112,179.0,male,no
458,S1048,96,172.0,male,yes
459,S1352,62,NaN,female,no


In [132]:
dfUS["height"] = dfUS["height"].round(0).astype("Int64") # Same thing again to get rid of decimals
dfUS["weight"] = dfUS["weight"].round(0).astype("Int64") # Couldn't see it in weight initially but no harm applying it anyways just in case
dfUS["BMI"] = (dfUS.weight)/(dfUS.height/100)**2 # Calculating BMI for US and adding to dataframe
dfUS # Looks good!

,id,weight,height,sex,smokes,BMI
0,S1272,61,<NA>,female,no,<NA>
1,S1288,105,<NA>,female,no,<NA>
2,S1361,92,<NA>,female,no,<NA>
3,S1091,78,182,male,no,23.547881
4,S1351,83,<NA>,female,no,<NA>
...,...,...,...,...,...,...
456,S1300,61,<NA>,female,no,<NA>
457,S1169,112,179,male,no,34.955214
458,S1048,96,172,male,yes,32.449973
459,S1352,62,<NA>,female,no,<NA>


## Preprocessing of UK data

In [133]:
dfUK["id"].is_unique # Unique ids.
dfUK

,id,weight,height,sex,smokes
0,UK_0001,180.8,-1.0,male,0
1,UK_0002,141.1,65.7,male,0
2,UK_0003,180.8,68.5,male,0
3,UK_0004,218.3,68.9,male,0
4,UK_0005,147.7,62.6,female,0
...,...,...,...,...,...
689,UK_0690,156.5,61.8,female,0
690,UK_0691,176.4,65.4,male,1
691,UK_0692,205.0,72.4,male,0
692,UK_0693,130.1,66.1,female,0


For the UK: Weight seems quite high in general, also in datawrangler so possible that this is in pounds instead of kilogram. Height is most certainly in inches instead of meter or centimer. The smoking status is also coded as 0 and 1 instead of yes no and for sex we have 7 distinct possibilities instead of 2 which we will have to fix. Finally in weight we have several individuals in the 950-1000 range. If there was one it could be a possibility, but they all seem to have a weight of 999 so I have a strong suspicion that these individuals just had missing values which has been filled in as 999. Similarily for height, some individuals have a negative height which I take to just be missing. Lets fix it!

In [134]:
dfUK.smokes = dfUK.smokes.astype(str) # Convert from integer to object just like the dtype in the polish set so I can change smoking status variable
dfUK.loc[(dfUK.smokes == "0"), "smokes"] = "no" # Convert the smoking variable to the dutch standard
dfUK.loc[(dfUK.smokes == "1"), "smokes"] = "yes"

In [135]:
dfUK["sex"].unique() # These are the unique values for sex column, lets dichotomize it

dfUK.loc[(dfUK.sex == "FEMALE"), "sex"] = "female"
dfUK.loc[(dfUK.sex == "Woman"), "sex"] = "female"
dfUK.loc[(dfUK.sex == "Male"), "sex"] = "male"
dfUK.loc[(dfUK.sex == "MALE"), "sex"] = "male"
dfUK.loc[(dfUK.sex == "Female"), "sex"] = "female"

In [136]:
dfUK["height"] = dfUK["height"]*2.54     # Inches to centimeters
dfUK["weight"] = dfUK["weight"]*0.453592 # Pounds to kilogram

In [137]:
dfUK["height"] = dfUK["height"].round(0).astype("Int64") # Same thing again to get rid of decimals
dfUK["weight"] = dfUK["weight"].round(0).astype("Int64")

In [138]:
dfUK.loc[(dfUK.height <= 0), "height"] = pd.NA
dfUK.loc[(dfUK.weight >= 450), "weight"] = pd.NA

In [139]:
dfUK["BMI"] = (dfUK.weight)/(dfUK.height/100)**2
dfUK # Looks good!

,id,weight,height,sex,smokes,BMI
0,UK_0001,82,<NA>,male,no,<NA>
1,UK_0002,64,167,male,no,22.948116
2,UK_0003,82,174,male,no,27.084159
3,UK_0004,99,175,male,no,32.326531
4,UK_0005,67,159,female,no,26.502116
...,...,...,...,...,...,...
689,UK_0690,71,157,female,no,28.804414
690,UK_0691,80,166,male,yes,29.03179
691,UK_0692,93,184,male,no,27.469282
692,UK_0693,59,168,female,no,20.904195


In [140]:
#dfNL.dtypes Can check one by one that the object types match for each dataframe.
#dfUS.dtypes
#dfPL.dtypes
#dfUK.dtypes

The data for each country has now been processed so that they match. The only remaining issue are the missing values, however for the descriptive initial analysis we will be performing I believe it is probably best to leave the missing values as is instead of dropping rows of individuals with any missing data in any of the variables. The reason for this I believe missing data mainly becomes an issue in statistical analysis and testing, for just descriptive analysis dropping an individual just because they have a missing value in height means we are now getting rid of the information we had on him regarding weight or smoking as well. Furthermore, leaving the missing data in allows people looking into the dataframes to be aware of the fact that the data from these countries are not complete which could be important information to have before jumping into further analysis.

## Merging the data

In [141]:
dfNL["source"] = "NL"
dfPL["source"] = "PL"
dfUS["source"] = "US"
dfUK["source"] = "GB"
dfmerged = pd.concat([dfNL, dfPL, dfUS, dfUK], axis = 0)
dfmerged.to_csv("merged_countrydata.csv", index=False)
dfmerged

,id,weight,height,sex,smokes,BMI,source
0,NL_0001,105,188,male,yes,29.708013,NL
1,NL_0002,114,177,male,yes,36.388011,NL
2,NL_0003,70,176,female,no,22.59814,NL
3,NL_0004,68,175,male,no,22.204082,NL
4,NL_0005,86,186,male,yes,24.858365,NL
...,...,...,...,...,...,...,...
689,UK_0690,71,157,female,no,28.804414,GB
690,UK_0691,80,166,male,yes,29.03179,GB
691,UK_0692,93,184,male,no,27.469282,GB
692,UK_0693,59,168,female,no,20.904195,GB


In [142]:
dfgenes

,id,source,geneA,geneB,geneC
0,NL_0590,NL,0.725787,10.516221,88.803319
1,NL_0395,NL,1.827190,-1.364823,100.720557
2,NL_0183,NL,2.701437,14.623456,89.894749
3,NL_0308,NL,1.729680,10.100416,99.305338
4,NL_0826,NL,0.470720,-3.894969,103.650732
...,...,...,...,...,...
3492,S1406,US,1.129991,8.910793,98.230566
3493,S1221,US,1.688210,7.187131,112.130364
3494,S1218,US,1.084599,-3.339440,121.564776
3495,S1306,US,27.838815,22.259590,93.332950


I have fewer rowes in the genes data than I do in the merged country data. I can only assume that this means that for some individuals there is no collected data on gene expression? In this case I believe instead of pd.concat I will have to merge the dataframes using the unique ids and matching source or countries that are in both dataframes using inner as method. The reason for this is because the two different dataframes are not perfectly matching on both the column variables and the ids so I couldnt figure out how to match them using pd.concat. Instead we use the merge method that gives us the final dataframe with all the information.

In [ ]:
dfFinal = dfmerged.merge(dfgenes, on=["id", "source"], how="inner")
dfFinal.to_csv("full_merged_data .csv", index=False)
dfFinal

,id,weight,height,sex,smokes,BMI,source,geneA,geneB,geneC
0,NL_0001,105,188,male,yes,29.708013,NL,2.852573,15.139682,88.698757
1,NL_0002,114,177,male,yes,36.388011,NL,23.406720,23.477106,105.267242
2,NL_0003,70,176,female,no,22.59814,NL,0.375769,-6.081572,99.667734
3,NL_0004,68,175,male,no,22.204082,NL,0.800761,-10.197789,82.935411
4,NL_0005,86,186,male,yes,24.858365,NL,1.008053,2.101691,97.957148
...,...,...,...,...,...,...,...,...,...,...
3487,UK_0690,71,157,female,no,28.804414,GB,2.777172,-2.525593,103.881438
3488,UK_0691,80,166,male,yes,29.03179,GB,3.933670,3.776257,98.917296
3489,UK_0692,93,184,male,no,27.469282,GB,1.335294,8.889390,107.504885
3490,UK_0693,59,168,female,no,20.904195,GB,0.203390,-17.683156,88.966823
